# 🐦 Twitter Sentiment Analysis using NLP & Machine Learning

**Author:** Kajal Gupta  
**GitHub:** [kajalvsg](https://github.com/kajalvsg)  

---

## 📌 Project Overview

This project builds a complete **NLP-based sentiment classification pipeline** on Twitter data.  
We classify tweets as **Positive** or **Negative** using:

- Text preprocessing (tokenization, stopword removal, lemmatization)
- TF-IDF vectorization
- Logistic Regression and Naive Bayes classifiers
- Evaluation using Precision, Recall, F1-Score, and Confusion Matrix

---

## 📂 Project Structure

```
sentiment-analysis/
├── notebooks/
│   └── sentiment_analysis.ipynb   ← You are here
├── data/
│   └── twitter_sentiment.csv      ← Dataset
├── outputs/
│   └── model_results.csv          ← Saved metrics
├── images/                        ← All visualizations
├── requirements.txt
└── README.md
```

## 1. 📦 Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score, precision_score, recall_score
)

print('All libraries imported successfully!')

## 2. 📊 Load & Explore Dataset

In [ ]:
# Load the dataset
df = pd.read_csv('../data/twitter_sentiment.csv')

print(f'Dataset Shape: {df.shape}')
print(f'Positive tweets: {df["label"].sum()}')
print(f'Negative tweets: {(df["label"]==0).sum()}')
df.head()

In [ ]:
# Class distribution plot
fig, ax = plt.subplots(figsize=(7, 4.5))
counts = df['sentiment'].value_counts()
bars = ax.bar(counts.index, counts.values,
              color=['#2ecc71', '#e74c3c'], width=0.45,
              edgecolor='white', linewidth=1.5)
for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            str(val), ha='center', fontsize=13, fontweight='bold')
ax.set_title('Class Distribution', fontsize=15, fontweight='bold')
ax.set_xlabel('Sentiment')
ax.set_ylabel('Number of Tweets')
ax.yaxis.grid(True, alpha=0.4)
ax.set_axisbelow(True)
plt.tight_layout()
plt.show()

## 3. 🧹 NLP Preprocessing

We apply the following steps to clean each tweet:

| Step | Description |
|------|-------------|
| Lowercasing | Convert all text to lowercase |
| URL Removal | Strip `http://` and `https://` links |
| Mention Removal | Remove `@username` patterns |
| Hashtag Cleaning | Strip `#` but keep the word |
| Punctuation Removal | Keep only alphabetic characters |
| Stopword Removal | Remove common English stopwords |
| Lemmatization | Reduce words to their base form |

In [ ]:
# Stopwords list (no NLTK download required)
STOPWORDS = {
    'i','me','my','myself','we','our','ours','ourselves','you','your','yours',
    'yourself','yourselves','he','him','his','himself','she','her','hers',
    'herself','it','its','itself','they','them','their','theirs','themselves',
    'what','which','who','whom','this','that','these','those','am','is','are',
    'was','were','be','been','being','have','has','had','having','do','does',
    'did','doing','a','an','the','and','but','if','or','because','as','until',
    'while','of','at','by','for','with','about','against','between','into',
    'through','during','before','after','above','below','to','from','up','down',
    'in','out','on','off','over','under','again','further','then','once','here',
    'there','when','where','why','how','all','both','each','few','more','most',
    'other','some','such','no','nor','not','only','own','same','so','than','too',
    'very','s','t','can','will','just','don','should','now','d','ll','m','o',
    're','ve','y','ain','aren','couldn','didn','doesn','hadn','hasn','haven',
    'isn','ma','mightn','mustn','needn','shan','shouldn','wasn','weren','won','wouldn',
}

def simple_lemmatize(word):
    """Rule-based lemmatizer (no NLTK download required)."""
    if word.endswith('ing') and len(word) > 5:
        return word[:-3]
    if word.endswith('ed') and len(word) > 4:
        return word[:-2]
    if word.endswith('ly') and len(word) > 4:
        return word[:-2]
    if word.endswith('ies') and len(word) > 4:
        return word[:-3] + 'y'
    if word.endswith('es') and len(word) > 3:
        return word[:-1]
    if word.endswith('s') and len(word) > 3 and not word.endswith('ss'):
        return word[:-1]
    return word

def preprocess_tweet(text):
    """Full NLP preprocessing pipeline for a single tweet."""
    text = str(text).lower()                          # 1. Lowercase
    text = re.sub(r'http\S+|www\S+', '', text)        # 2. Remove URLs
    text = re.sub(r'@\w+', '', text)                  # 3. Remove @mentions
    text = re.sub(r'#(\w+)', r'\1', text)             # 4. Clean hashtags
    text = re.sub(r'[^a-z\s]', '', text)              # 5. Remove punctuation
    tokens = text.split()                              # 6. Tokenize
    tokens = [simple_lemmatize(t)                      # 7. Lemmatize
              for t in tokens
              if t not in STOPWORDS and len(t) > 2]   # 8. Remove stopwords
    return ' '.join(tokens)

df['clean_tweet'] = df['tweet'].apply(preprocess_tweet)

print('Preprocessing complete!\n')
print('Before:', df['tweet'].iloc[0])
print('After: ', df['clean_tweet'].iloc[0])

## 4. 🔢 TF-IDF Vectorization

In [ ]:
X = df['clean_tweet']
y = df['label']

# 80/20 stratified split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# TF-IDF with unigrams + bigrams
tfidf = TfidfVectorizer(max_features=3000, ngram_range=(1, 2), min_df=1)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)

print(f'Training set size : {X_train_tfidf.shape}')
print(f'Test set size     : {X_test_tfidf.shape}')
print(f'Vocabulary size   : {len(tfidf.vocabulary_)} unique tokens')

## 5. 🤖 Train Models

We train two baseline classifiers:
- **Logistic Regression** – strong linear baseline for text
- **Naive Bayes (Multinomial)** – probabilistic model well-suited for TF-IDF features

In [ ]:
# --- Logistic Regression ---
lr = LogisticRegression(max_iter=1000, random_state=42, C=1.0)
lr.fit(X_train_tfidf, y_train)
lr_pred = lr.predict(X_test_tfidf)

# --- Naive Bayes ---
nb = MultinomialNB(alpha=0.5)
nb.fit(X_train_tfidf, y_train)
nb_pred = nb.predict(X_test_tfidf)

print('Models trained!')

## 6. 📈 Evaluation

In [ ]:
def get_metrics(y_true, y_pred, name):
    return {
        'Model':     name,
        'Accuracy':  round(accuracy_score(y_true, y_pred)*100, 2),
        'Precision': round(precision_score(y_true, y_pred)*100, 2),
        'Recall':    round(recall_score(y_true, y_pred)*100, 2),
        'F1-Score':  round(f1_score(y_true, y_pred)*100, 2),
    }

results = pd.DataFrame([
    get_metrics(y_test, lr_pred, 'Logistic Regression'),
    get_metrics(y_test, nb_pred, 'Naive Bayes'),
])

print('=== Model Performance (%) ===')
results

In [ ]:
# Full classification report
print('--- Logistic Regression ---')
print(classification_report(y_test, lr_pred, target_names=['Negative','Positive']))

print('--- Naive Bayes ---')
print(classification_report(y_test, nb_pred, target_names=['Negative','Positive']))

In [ ]:
# Confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
fig.suptitle('Confusion Matrices', fontsize=15, fontweight='bold')

for ax, pred, name, clr in zip(axes,
    [lr_pred, nb_pred],
    ['Logistic Regression', 'Naive Bayes'],
    ['#3498db', '#9b59b6']):
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(cm, annot=True, fmt='d',
                cmap=sns.light_palette(clr, as_cmap=True),
                linewidths=1, linecolor='white',
                xticklabels=['Negative','Positive'],
                yticklabels=['Negative','Positive'],
                ax=ax, annot_kws={'size':14,'weight':'bold'})
    ax.set_title(name, fontsize=13, fontweight='bold')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.tight_layout()
plt.show()

In [ ]:
# Model comparison bar chart
metrics_cols = ['Accuracy','Precision','Recall','F1-Score']
x = np.arange(len(metrics_cols))
width = 0.32

fig, ax = plt.subplots(figsize=(9, 5))
lr_vals = results.loc[results['Model']=='Logistic Regression', metrics_cols].values[0]
nb_vals = results.loc[results['Model']=='Naive Bayes',         metrics_cols].values[0]

b1 = ax.bar(x - width/2, lr_vals, width, label='Logistic Regression',
            color='#3498db', edgecolor='white', linewidth=1.2)
b2 = ax.bar(x + width/2, nb_vals, width, label='Naive Bayes',
            color='#9b59b6', edgecolor='white', linewidth=1.2)

for bar in list(b1) + list(b2):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.4,
            f'{bar.get_height():.1f}%', ha='center', va='bottom',
            fontsize=9, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(metrics_cols, fontsize=12)
ax.set_ylabel('Score (%)')
ax.set_title('Model Performance Comparison', fontsize=15, fontweight='bold')
ax.set_ylim(0, 115)
ax.yaxis.grid(True, alpha=0.4)
ax.set_axisbelow(True)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# Top TF-IDF features by LR coefficient
feature_names = np.array(tfidf.get_feature_names_out())
lr_coef = lr.coef_[0]

top_pos_idx = lr_coef.argsort()[-15:][::-1]
top_neg_idx = lr_coef.argsort()[:15]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5.5))
fig.suptitle('Top 15 Influential TF-IDF Features', fontsize=14, fontweight='bold')

for ax, idx, color, title in [
    (ax1, top_pos_idx, '#2ecc71', 'Top Positive Words'),
    (ax2, top_neg_idx, '#e74c3c', 'Top Negative Words'),
]:
    words  = feature_names[idx][::-1]
    scores = np.abs(lr_coef[idx])[::-1]
    ax.barh(words, scores, color=color, edgecolor='white')
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel('|Coefficient|')
    ax.xaxis.grid(True, alpha=0.4)
    ax.set_axisbelow(True)

plt.tight_layout()
plt.show()

In [ ]:
# Tweet length distribution
df['tweet_length'] = df['tweet'].apply(lambda x: len(x.split()))

fig, ax = plt.subplots(figsize=(8, 4.5))
for sentiment, color, label in [('positive','#2ecc71','Positive'),
                                  ('negative','#e74c3c','Negative')]:
    subset = df[df['sentiment'] == sentiment]['tweet_length']
    ax.hist(subset, bins=12, alpha=0.65, color=color,
            label=label, edgecolor='white')

ax.set_title('Tweet Length Distribution by Sentiment', fontsize=14, fontweight='bold')
ax.set_xlabel('Number of Words')
ax.set_ylabel('Frequency')
ax.yaxis.grid(True, alpha=0.4)
ax.set_axisbelow(True)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

## 7. 🔍 Predict on Custom Tweets

In [ ]:
def predict_sentiment(tweet, model=lr):
    """Predict sentiment for any custom tweet."""
    cleaned = preprocess_tweet(tweet)
    vectorized = tfidf.transform([cleaned])
    pred = model.predict(vectorized)[0]
    prob = model.predict_proba(vectorized)[0]
    label = 'POSITIVE 😊' if pred == 1 else 'NEGATIVE 😞'
    confidence = round(max(prob) * 100, 2)
    print(f'Tweet      : {tweet}')
    print(f'Sentiment  : {label}')
    print(f'Confidence : {confidence}%\n')

# Try it out!
predict_sentiment('I absolutely love this new phone, it is amazing!')
predict_sentiment('This is the worst service I have ever experienced.')
predict_sentiment('The weather is okay today, nothing special.')

## 8. 💾 Save Results

In [ ]:
results.to_csv('../outputs/model_results.csv', index=False)
print('Results saved to outputs/model_results.csv')
print('\nFinal Summary:')
print(results.to_string(index=False))

---

## ✅ Summary

| Step | Details |
|------|---------|
| Dataset | 100 Twitter-style tweets (50 positive, 50 negative) |
| Preprocessing | Lowercasing, URL/mention removal, stopword removal, lemmatization |
| Vectorization | TF-IDF with unigrams + bigrams, max 3000 features |
| Models | Logistic Regression, Multinomial Naive Bayes |
| Best Accuracy | 70% on 20-tweet test set |
| Key Insight | Both models perform comparably on this balanced dataset |

**Built by Kajal Gupta** | [LinkedIn](https://www.linkedin.com/in/kajal-gupta-a18908246) | [GitHub](https://github.com/kajalvsg)